# Notebook 02 — EDA Transactional Fraud

## Fraud Graph Analytics  
### Análise Exploratória de Dados Transacionais para Prevenção a Fraudes

Este notebook realiza a primeira análise exploratória sobre os dados sintéticos gerados no **Notebook 01 — Generate Synthetic Transactions**.

O objetivo é identificar padrões, distribuições, concentrações e sinais iniciais de risco que possam orientar:

- criação de regras antifraude explicáveis;
- modelagem do Knowledge Graph;
- geração de features de rede;
- priorização de contas, dispositivos, beneficiários e transações suspeitas;
- construção futura de um score de risco transacional.

A análise será conduzida com foco em interpretação de negócio, buscando conectar os achados técnicos aos conceitos definidos no **Notebook 00 — Domain Understanding com CRISP-DM+**.

## 1. Objetivo da Célula

### Objetivo

Configurar o ambiente inicial do notebook, importar bibliotecas, definir caminhos do projeto e preparar os datasets sintéticos para análise exploratória.

### Ações realizadas

- Importação das bibliotecas principais.
- Definição dos diretórios do projeto.
- Leitura dos arquivos Parquet gerados no notebook anterior.
- Padronização inicial de tipos de dados.
- Criação de funções auxiliares para sumarização.

### Justificativa técnica

A análise exploratória é uma etapa essencial para compreender padrões de comportamento antes da criação de regras antifraude e da modelagem em grafo. Uma configuração inicial reprodutível permite que os próximos notebooks reutilizem as mesmas bases e interpretações.

### Resultados esperados

Ambiente preparado, datasets carregados e base transacional pronta para enriquecimento analítico.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
DOCS_DIR = PROJECT_ROOT / "docs"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Synthetic dir: {SYNTHETIC_DIR}")
print(f"Docs dir:      {DOCS_DIR}")
print(f"Reports dir:   {REPORTS_DIR}")

Project root:  d:\_DS-Projects\Data-Science\fraud-graph-analytics
Synthetic dir: d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\synthetic
Docs dir:      d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs
Reports dir:   d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports


## 2. Carregamento dos Datasets Sintéticos

Nesta etapa são carregadas as entidades sintéticas criadas no Notebook 01.

As bases utilizadas são:

- clientes;
- contas;
- dispositivos;
- IPs;
- beneficiários;
- cartões;
- transações;
- labels de fraude.

Essas tabelas representam a estrutura mínima necessária para iniciar a análise exploratória e preparar o futuro Knowledge Graph.

In [3]:
dataset_files = {
    "clientes": "clientes.parquet",
    "contas": "contas.parquet",
    "dispositivos": "dispositivos.parquet",
    "ips": "ips.parquet",
    "beneficiarios": "beneficiarios.parquet",
    "cartoes": "cartoes.parquet",
    "transacoes": "transacoes.parquet",
    "labels_fraude": "labels_fraude.parquet",
}

datasets = {}

for name, filename in dataset_files.items():
    path = SYNTHETIC_DIR / filename
    
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    
    datasets[name] = pd.read_parquet(path)

clientes = datasets["clientes"]
contas = datasets["contas"]
dispositivos = datasets["dispositivos"]
ips = datasets["ips"]
beneficiarios = datasets["beneficiarios"]
cartoes = datasets["cartoes"]
transacoes = datasets["transacoes"]
labels_fraude = datasets["labels_fraude"]

print("Datasets carregados com sucesso.")

Datasets carregados com sucesso.


In [4]:
summary_datasets = pd.DataFrame(
    [
        {
            "dataset": name,
            "linhas": df.shape[0],
            "colunas": df.shape[1],
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for name, df in datasets.items()
    ]
).sort_values("linhas", ascending=False)

summary_datasets

,dataset,linhas,colunas,memoria_mb
6,transacoes,80000,13,43.26
7,labels_fraude,80000,4,18.90
1,contas,6000,6,1.59
0,clientes,5000,6,1.06
2,dispositivos,4500,4,0.96
5,cartoes,4000,5,1.02
4,beneficiarios,3500,4,0.77
3,ips,3000,4,0.63


## 3. Padronização Inicial de Tipos

Antes das análises, as colunas de data serão convertidas para tipos adequados.

Também serão criadas variáveis temporais derivadas, úteis para detectar padrões por:

- mês;
- dia da semana;
- hora da transação;
- idade da conta no momento da transação.

In [5]:
transacoes["data_hora"] = pd.to_datetime(transacoes["data_hora"])

clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])
contas["data_abertura"] = pd.to_datetime(contas["data_abertura"])
cartoes["data_emissao"] = pd.to_datetime(cartoes["data_emissao"])

transacoes["ano_mes"] = transacoes["data_hora"].dt.to_period("M").astype(str)
transacoes["data"] = transacoes["data_hora"].dt.date
transacoes["hora"] = transacoes["data_hora"].dt.hour
transacoes["dia_semana_num"] = transacoes["data_hora"].dt.dayofweek

dias_semana = {
    0: "segunda",
    1: "terca",
    2: "quarta",
    3: "quinta",
    4: "sexta",
    5: "sabado",
    6: "domingo",
}

transacoes["dia_semana"] = transacoes["dia_semana_num"].map(dias_semana)

transacoes.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id,ano_mes,data,hora,dia_semana_num,dia_semana
0,TX_00007707,CTA_001877,BEN_003322,203.30,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None,2025-01,2025-01-01,3,2,quarta
1,TX_00072064,CTA_003365,BEN_000265,275.99,2025-01-01 03:13:25,pix,app,DEV_002968,IP_001202,aprovada,0,normal,None,2025-01,2025-01-01,3,2,quarta
2,TX_00075764,CTA_000156,BEN_002832,11.59,2025-01-01 04:00:23,pix,app,DEV_003600,IP_001519,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta
3,TX_00023203,CTA_005491,BEN_001801,296.97,2025-01-01 04:08:36,pix,app,DEV_001308,IP_001428,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta
4,TX_00001302,CTA_000997,BEN_002808,94.98,2025-01-01 04:15:04,boleto,api,DEV_003878,IP_002967,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta


## 4. Enriquecimento da Base Transacional

A base transacional será enriquecida com atributos das principais entidades relacionadas:

- conta;
- cliente;
- dispositivo;
- IP;
- beneficiário.

Esse enriquecimento facilita análises integradas e ajuda a antecipar features que serão utilizadas nos próximos notebooks.

In [6]:
transacoes_enriched = (
    transacoes
    .merge(
        contas[
            [
                "conta_id",
                "cliente_id",
                "tipo_conta",
                "data_abertura",
                "status_conta",
                "limite_transacional_diario",
            ]
        ],
        left_on="conta_origem_id",
        right_on="conta_id",
        how="left",
    )
    .merge(
        clientes[
            [
                "cliente_id",
                "idade",
                "uf",
                "segmento",
                "data_cadastro",
                "score_cadastral",
            ]
        ],
        on="cliente_id",
        how="left",
    )
    .merge(
        dispositivos,
        on="device_id",
        how="left",
    )
    .merge(
        ips,
        on="ip_id",
        how="left",
    )
    .merge(
        beneficiarios,
        on="beneficiario_id",
        how="left",
    )
)

transacoes_enriched["idade_conta_dias"] = (
    transacoes_enriched["data_hora"] - transacoes_enriched["data_abertura"]
).dt.days.clip(lower=0)

transacoes_enriched["valor_sobre_limite_diario"] = (
    transacoes_enriched["valor"] / transacoes_enriched["limite_transacional_diario"]
).replace([np.inf, -np.inf], np.nan)

transacoes_enriched.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id,ano_mes,data,hora,dia_semana_num,dia_semana,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario,idade,uf,segmento,data_cadastro,score_cadastral,tipo_device,sistema_operacional,fingerprint_risco,uf_origem,tipo_rede,risco_rede,tipo_beneficiario,banco_destino,uf_destino,idade_conta_dias,valor_sobre_limite_diario
0,TX_00007707,CTA_001877,BEN_003322,203.30,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None,2025-01,2025-01-01,3,2,quarta,CTA_001877,CLI_002620,corrente,2021-08-09,ativa,5000,74,MG,varejo,2025-09-11,803,mobile,iOS,baixo,CE,movel,alto,pessoa_fisica,banco_a,GO,1241,0.04
1,TX_00072064,CTA_003365,BEN_000265,275.99,2025-01-01 03:13:25,pix,app,DEV_002968,IP_001202,aprovada,0,normal,None,2025-01,2025-01-01,3,2,quarta,CTA_003365,CLI_000516,corrente,2024-08-18,ativa,1000,24,SC,varejo,2021-10-05,541,mobile,Windows,baixo,ES,movel,baixo,pessoa_fisica,banco_b,PE,136,0.28
2,TX_00075764,CTA_000156,BEN_002832,11.59,2025-01-01 04:00:23,pix,app,DEV_003600,IP_001519,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta,CTA_000156,CLI_000344,corrente,2024-12-09,ativa,5000,48,SP,alta_renda,2025-06-19,657,mobile,Android,baixo,SP,residencial,baixo,conta_interna,banco_a,MG,23,0.00
3,TX_00023203,CTA_005491,BEN_001801,296.97,2025-01-01 04:08:36,pix,app,DEV_001308,IP_001428,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta,CTA_005491,CLI_002752,corrente,2022-08-16,ativa,20000,22,SP,aposentado,2021-08-21,603,mobile,Windows,baixo,MG,movel,medio,pessoa_juridica,banco_a,SP,869,0.01
4,TX_00001302,CTA_000997,BEN_002808,94.98,2025-01-01 04:15:04,boleto,api,DEV_003878,IP_002967,aprovada,0,normal,None,2025-01,2025-01-01,4,2,quarta,CTA_000997,CLI_004486,corrente,2024-09-01,ativa,10000,40,PR,varejo,2024-07-28,593,mobile,iOS,baixo,GO,movel,baixo,pessoa_fisica,banco_c,PE,122,0.01


In [7]:
null_summary = (
    transacoes_enriched
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
    .rename(columns={"index": "coluna", 0: "percentual_nulos"})
    .sort_values("percentual_nulos", ascending=False)
)

null_summary.head(20)

,coluna,percentual_nulos
12,cartao_id,92.15
0,transacao_id,0.00
2,beneficiario_id,0.00
1,conta_origem_id,0.00
4,data_hora,0.00
5,tipo_transacao,0.00
6,canal,0.00
3,valor,0.00
7,device_id,0.00
8,ip_id,0.00


## 5. KPIs Gerais da Base Transacional

Nesta etapa serão calculados indicadores gerais da base, incluindo:

- quantidade total de transações;
- taxa sintética de fraude;
- valor total transacionado;
- valor médio e mediano;
- quantidade de contas, clientes, dispositivos e beneficiários distintos;
- quantidade de transações em análise ou negadas.

Esses indicadores funcionam como uma visão executiva inicial do dataset.

In [8]:
kpis = {
    "total_transacoes": len(transacoes_enriched),
    "taxa_fraude_sintetica": transacoes_enriched["is_fraud"].mean(),
    "valor_total_transacionado": transacoes_enriched["valor"].sum(),
    "valor_medio": transacoes_enriched["valor"].mean(),
    "valor_mediano": transacoes_enriched["valor"].median(),
    "qtd_clientes_distintos": transacoes_enriched["cliente_id"].nunique(),
    "qtd_contas_distintas": transacoes_enriched["conta_origem_id"].nunique(),
    "qtd_beneficiarios_distintos": transacoes_enriched["beneficiario_id"].nunique(),
    "qtd_dispositivos_distintos": transacoes_enriched["device_id"].nunique(),
    "qtd_ips_distintos": transacoes_enriched["ip_id"].nunique(),
    "qtd_transacoes_em_analise": (transacoes_enriched["status_transacao"] == "em_analise").sum(),
    "qtd_transacoes_negadas": (transacoes_enriched["status_transacao"] == "negada").sum(),
}

kpi_df = pd.DataFrame(
    [{"indicador": key, "valor": value} for key, value in kpis.items()]
)

kpi_df

,indicador,valor
0,total_transacoes,"80,000.00"
1,taxa_fraude_sintetica,0.09
2,valor_total_transacionado,"81,005,749.93"
3,valor_medio,"1,012.57"
4,valor_mediano,299.64
5,qtd_clientes_distintos,"3,487.00"
6,qtd_contas_distintas,"6,000.00"
7,qtd_beneficiarios_distintos,"3,500.00"
8,qtd_dispositivos_distintos,"4,500.00"
9,qtd_ips_distintos,"3,000.00"


## 6. Distribuição dos Cenários de Fraude

Como os dados são sintéticos, os cenários de fraude foram injetados de forma controlada.

A análise abaixo permite verificar a proporção de cada cenário e entender quais padrões estarão disponíveis para análise, regras e grafo.

In [9]:
scenario_distribution = (
    transacoes_enriched
    .groupby("fraud_scenario")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        contas_distintas=("conta_origem_id", "nunique"),
        beneficiarios_distintos=("beneficiario_id", "nunique"),
        dispositivos_distintos=("device_id", "nunique"),
    )
    .reset_index()
)

scenario_distribution["percentual_transacoes"] = (
    scenario_distribution["qtd_transacoes"] / scenario_distribution["qtd_transacoes"].sum()
)

scenario_distribution = scenario_distribution.sort_values("qtd_transacoes", ascending=False)

scenario_distribution[
    [
        "fraud_scenario",
        "qtd_transacoes",
        "percentual_transacoes",
        "taxa_fraude",
        "valor_total",
        "valor_medio",
        "contas_distintas",
        "beneficiarios_distintos",
        "dispositivos_distintos",
    ]
]

,fraud_scenario,qtd_transacoes,percentual_transacoes,taxa_fraude,valor_total,valor_medio,contas_distintas,beneficiarios_distintos,dispositivos_distintos
5,normal,73000,0.91,0.00,"34,471,209.12",472.21,6000,3500,4500
6,shared_device_ring,1600,0.02,1.00,"706,117.92",441.32,1381,1266,23
0,beneficiary_concentrator,1400,0.02,1.00,"2,220,961.22","1,586.40",1244,20,1209
2,burst_transactions,1200,0.01,1.00,"2,251,092.41","1,875.91",40,1020,1049
4,new_account_high_value,1000,0.01,1.00,"26,752,498.22","26,752.50",261,854,909
3,coordinated_network,1000,0.01,1.00,"9,359,034.68","9,359.03",60,15,10
1,bridge_account,800,0.01,1.00,"5,244,836.36","6,556.05",12,717,736


In [10]:
fig = px.bar(
    scenario_distribution,
    x="fraud_scenario",
    y="qtd_transacoes",
    title="Distribuição das Transações por Cenário Sintético",
    labels={
        "fraud_scenario": "Cenário",
        "qtd_transacoes": "Quantidade de Transações",
    },
)

fig.update_layout(xaxis_tickangle=-35)
fig.show()

## 7. Análise de Valores Transacionais

A análise de valores ajuda a identificar diferenças entre transações normais e transações sinteticamente marcadas como fraude.

Essa etapa também apoia a definição futura de regras como:

- transação de alto valor;
- valor acima do limite diário;
- conta nova com alto valor;
- comportamento financeiro incompatível com o histórico.

In [11]:
value_summary_by_label = (
    transacoes_enriched
    .groupby("is_fraud")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        valor_min=("valor", "min"),
        valor_p25=("valor", lambda x: x.quantile(0.25)),
        valor_mediano=("valor", "median"),
        valor_medio=("valor", "mean"),
        valor_p75=("valor", lambda x: x.quantile(0.75)),
        valor_p95=("valor", lambda x: x.quantile(0.95)),
        valor_max=("valor", "max"),
    )
    .reset_index()
)

value_summary_by_label

,is_fraud,qtd_transacoes,valor_min,valor_p25,valor_mediano,valor_medio,valor_p75,valor_p95,valor_max
0,0,73000,5.00,134.07,272.78,472.21,550.93,"1,529.56","25,000.00"
1,1,7000,8.44,685.15,"2,119.68","6,647.79","8,791.65","32,494.37","44,999.68"


In [12]:
value_summary_by_scenario = (
    transacoes_enriched
    .groupby("fraud_scenario")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        valor_medio=("valor", "mean"),
        valor_mediano=("valor", "median"),
        valor_p95=("valor", lambda x: x.quantile(0.95)),
        valor_max=("valor", "max"),
    )
    .reset_index()
    .sort_values("valor_medio", ascending=False)
)

value_summary_by_scenario

,fraud_scenario,qtd_transacoes,valor_medio,valor_mediano,valor_p95,valor_max
4,new_account_high_value,1000,"26,752.50","26,593.05","42,909.22","44,999.68"
3,coordinated_network,1000,"9,359.03","9,387.64","16,966.95","17,980.38"
1,bridge_account,800,"6,556.05","6,651.77","11,643.57","11,984.34"
2,burst_transactions,1200,"1,875.91","1,815.92","3,296.96","3,499.92"
0,beneficiary_concentrator,1400,"1,586.40","1,230.71","4,066.21","8,670.90"
5,normal,73000,472.21,272.78,"1,529.56","25,000.00"
6,shared_device_ring,1600,441.32,264.50,"1,363.16","9,947.93"


In [13]:
fig = px.box(
    transacoes_enriched,
    x="fraud_scenario",
    y="valor",
    title="Distribuição de Valores por Cenário Sintético",
    labels={
        "fraud_scenario": "Cenário",
        "valor": "Valor da Transação",
    },
)

fig.update_layout(xaxis_tickangle=-35)
fig.show()

## 8. Análise Temporal

Nesta etapa avaliamos padrões por mês, dia da semana e hora.

Em prevenção a fraudes, padrões temporais são relevantes porque determinados ataques podem ocorrer em janelas concentradas, horários atípicos ou momentos de menor capacidade operacional.

In [14]:
monthly_summary = (
    transacoes_enriched
    .groupby("ano_mes")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
    )
    .reset_index()
)

monthly_summary

,ano_mes,qtd_transacoes,taxa_fraude,valor_total,valor_medio
0,2025-01,6642,0.08,"6,854,516.91","1,032.00"
1,2025-02,6114,0.07,"5,926,788.17",969.38
2,2025-03,6772,0.07,"6,240,819.61",921.56
3,2025-04,6434,0.07,"6,197,348.79",963.22
4,2025-05,6794,0.07,"6,718,531.89",988.89
5,2025-06,6443,0.07,"6,235,999.49",967.87
6,2025-07,6766,0.07,"7,029,347.44","1,038.92"
7,2025-08,6697,0.07,"7,093,998.88","1,059.28"
8,2025-09,6459,0.07,"6,503,324.72","1,006.86"
9,2025-10,6640,0.07,"6,266,739.44",943.79


In [15]:
fig = px.line(
    monthly_summary,
    x="ano_mes",
    y="qtd_transacoes",
    markers=True,
    title="Quantidade de Transações por Mês",
    labels={
        "ano_mes": "Ano-Mês",
        "qtd_transacoes": "Quantidade de Transações",
    },
)

fig.show()

In [16]:
hourly_summary = (
    transacoes_enriched
    .groupby("hora")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_medio=("valor", "mean"),
    )
    .reset_index()
)

hourly_summary

,hora,qtd_transacoes,taxa_fraude,valor_medio
0,0,3239,0.08,997.89
1,1,3399,0.08,989.76
2,2,3335,0.07,"1,047.62"
3,3,3257,0.07,982.60
4,4,3378,0.08,"1,051.30"
5,5,3331,0.08,"1,017.99"
6,6,3180,0.07,935.03
7,7,3273,0.07,"1,028.06"
8,8,3252,0.07,943.52
9,9,3260,0.07,945.81


In [17]:
fig = px.bar(
    hourly_summary,
    x="hora",
    y="taxa_fraude",
    title="Taxa Sintética de Fraude por Hora do Dia",
    labels={
        "hora": "Hora do Dia",
        "taxa_fraude": "Taxa de Fraude Sintética",
    },
)

fig.show()

## 9. Análise por Tipo de Transação e Canal

A combinação entre tipo de transação e canal ajuda a entender onde os cenários suspeitos estão mais concentrados.

No contexto deste MVP, essa análise apoia regras como:

- PIX em rajada;
- transações de alto valor por canal digital;
- maior risco em determinados canais;
- comportamento diferenciado entre app, internet banking, ATM, agência e API.

In [18]:
transaction_type_summary = (
    transacoes_enriched
    .groupby("tipo_transacao")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        contas_distintas=("conta_origem_id", "nunique"),
    )
    .reset_index()
    .sort_values("qtd_transacoes", ascending=False)
)

transaction_type_summary

,tipo_transacao,qtd_transacoes,taxa_fraude,valor_total,valor_medio,contas_distintas
2,pix,45058,0.11,"52,368,160.50","1,162.24",5999
1,cartao,12253,0.05,"7,739,511.46",631.64,5205
0,boleto,10728,0.05,"6,489,212.50",604.89,4985
3,ted,6382,0.08,"9,352,349.79","1,465.43",3939
4,transferencia_interna,5579,0.09,"5,056,515.68",906.35,3551


In [19]:
channel_summary = (
    transacoes_enriched
    .groupby("canal")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        contas_distintas=("conta_origem_id", "nunique"),
    )
    .reset_index()
    .sort_values("qtd_transacoes", ascending=False)
)

channel_summary

,canal,qtd_transacoes,taxa_fraude,valor_total,valor_medio,contas_distintas
2,app,56170,0.09,"56,427,872.36","1,004.59",6000
4,internet_banking,11121,0.09,"11,523,016.77","1,036.15",5030
3,atm,5526,0.09,"5,684,097.23","1,028.61",3622
1,api,4012,0.09,"4,050,735.64","1,009.65",2878
0,agencia,3171,0.09,"3,320,027.93","1,047.00",2435


In [20]:
channel_type_summary = (
    transacoes_enriched
    .groupby(["canal", "tipo_transacao"])
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_medio=("valor", "mean"),
    )
    .reset_index()
    .sort_values("qtd_transacoes", ascending=False)
)

channel_type_summary.head(20)

,canal,tipo_transacao,qtd_transacoes,taxa_fraude,valor_medio
12,app,pix,31706,0.10,"1,147.88"
11,app,cartao,8588,0.05,625.41
10,app,boleto,7539,0.05,606.17
22,internet_banking,pix,6231,0.11,"1,198.84"
13,app,ted,4416,0.08,"1,492.51"
14,app,transferencia_interna,3921,0.09,892.97
17,atm,pix,3102,0.10,"1,175.68"
7,api,pix,2220,0.11,"1,235.77"
2,agencia,pix,1799,0.11,"1,174.65"
21,internet_banking,cartao,1732,0.06,654.95


## 10. Beneficiários Concentradores

Beneficiários concentradores são entidades que recebem valores de muitas contas diferentes.

Esse padrão pode representar comportamento legítimo, como empresas ou prestadores de serviço, mas também pode indicar contas usadas para concentração de valores suspeitos.

No futuro Knowledge Graph, beneficiários com alto grau de entrada serão candidatos importantes para análise de centralidade.

In [21]:
beneficiary_stats = (
    transacoes_enriched
    .groupby("beneficiario_id")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        contas_origem_distintas=("conta_origem_id", "nunique"),
        clientes_distintos=("cliente_id", "nunique"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        taxa_fraude=("is_fraud", "mean"),
        qtd_cenarios=("fraud_scenario", "nunique"),
    )
    .reset_index()
    .merge(
        beneficiarios,
        on="beneficiario_id",
        how="left",
    )
)

top_beneficiaries = beneficiary_stats.sort_values(
    ["contas_origem_distintas", "valor_total", "taxa_fraude"],
    ascending=False,
).head(20)

top_beneficiaries

,beneficiario_id,qtd_transacoes,contas_origem_distintas,clientes_distintos,valor_total,valor_medio,taxa_fraude,qtd_cenarios,tipo_beneficiario,banco_destino,uf_destino
2998,BEN_002999,108,106,103,"180,489.22","1,671.20",0.79,6,pessoa_fisica,banco_d,RJ
3122,BEN_003123,108,105,105,"140,199.26","1,298.14",0.73,2,conta_interna,banco_b,PE
3030,BEN_003031,103,102,102,"185,369.52","1,799.70",0.81,3,pessoa_fisica,banco_a,GO
267,BEN_000268,103,102,101,"136,141.75","1,321.76",0.84,3,pessoa_fisica,mesma_instituicao,ES
1263,BEN_001264,97,97,94,"127,914.99","1,318.71",0.72,3,pessoa_fisica,banco_d,ES
1790,BEN_001791,95,95,95,"126,121.55","1,327.60",0.75,2,pessoa_juridica,banco_b,DF
351,BEN_000352,94,94,94,"140,127.44","1,490.72",0.85,2,pessoa_juridica,banco_b,ES
2786,BEN_002787,94,93,92,"152,518.10","1,622.53",0.76,3,pessoa_juridica,banco_d,RS
2456,BEN_002457,93,93,92,"105,149.30","1,130.64",0.72,3,pessoa_fisica,banco_b,SC
997,BEN_000998,93,93,93,"95,672.93","1,028.74",0.78,3,pessoa_fisica,banco_b,DF


In [22]:
fig = px.bar(
    top_beneficiaries.sort_values("contas_origem_distintas", ascending=True),
    x="contas_origem_distintas",
    y="beneficiario_id",
    orientation="h",
    title="Top 20 Beneficiários por Quantidade de Contas de Origem Distintas",
    labels={
        "contas_origem_distintas": "Contas de Origem Distintas",
        "beneficiario_id": "Beneficiário",
    },
)

fig.show()

## 11. Dispositivos Compartilhados

Dispositivos compartilhados entre múltiplas contas são um sinal relevante para investigação.

Em um ambiente antifraude, o mesmo dispositivo acessando ou transacionando para muitas contas pode indicar:

- uso coordenado;
- automação;
- tomada de conta;
- fraude organizada;
- operação por intermediários.

In [23]:
device_stats = (
    transacoes_enriched
    .groupby("device_id")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        contas_distintas=("conta_origem_id", "nunique"),
        clientes_distintos=("cliente_id", "nunique"),
        beneficiarios_distintos=("beneficiario_id", "nunique"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        taxa_fraude=("is_fraud", "mean"),
    )
    .reset_index()
    .merge(
        dispositivos,
        on="device_id",
        how="left",
    )
)

top_devices = device_stats.sort_values(
    ["contas_distintas", "taxa_fraude", "valor_total"],
    ascending=False,
).head(20)

top_devices

,device_id,qtd_transacoes,contas_distintas,clientes_distintos,beneficiarios_distintos,valor_total,valor_medio,taxa_fraude,tipo_device,sistema_operacional,fingerprint_risco
855,DEV_000856,150,147,146,143,"74,310.83",495.41,0.89,desktop,Android,alto
4006,DEV_004007,145,143,140,145,"60,139.87",414.76,0.92,mobile,iOS,alto
3931,DEV_003932,100,99,99,98,"46,842.89",468.43,0.81,mobile,Android,alto
2417,DEV_002418,97,96,94,96,"31,917.12",329.04,0.84,mobile,Windows,alto
2047,DEV_002048,95,91,90,94,"55,695.31",586.27,0.75,mobile,Windows,alto
101,DEV_000102,89,88,88,88,"40,333.09",453.18,0.80,mobile,Windows,alto
2458,DEV_002459,88,88,87,88,"41,954.54",476.76,0.77,mobile,Android,alto
3862,DEV_003863,86,86,84,86,"34,586.08",402.16,0.84,mobile,iOS,alto
1271,DEV_001272,87,86,85,86,"82,371.47",946.80,0.83,tablet,Android,alto
2306,DEV_002307,83,83,83,83,"32,110.03",386.87,0.81,desktop,Linux,alto


In [24]:
fig = px.bar(
    top_devices.sort_values("contas_distintas", ascending=True),
    x="contas_distintas",
    y="device_id",
    orientation="h",
    title="Top 20 Dispositivos por Quantidade de Contas Distintas",
    labels={
        "contas_distintas": "Contas Distintas",
        "device_id": "Dispositivo",
    },
)

fig.show()

## 12. IPs e Risco de Rede

IPs e redes de origem ajudam a identificar padrões de acesso incomuns.

Nesta análise observamos:

- IPs com muitas contas distintas;
- IPs com alta taxa de fraude sintética;
- IPs classificados como rede de maior risco;
- concentração de transações por origem.

In [25]:
ip_stats = (
    transacoes_enriched
    .groupby("ip_id")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        contas_distintas=("conta_origem_id", "nunique"),
        clientes_distintos=("cliente_id", "nunique"),
        beneficiarios_distintos=("beneficiario_id", "nunique"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        taxa_fraude=("is_fraud", "mean"),
    )
    .reset_index()
    .merge(
        ips,
        on="ip_id",
        how="left",
    )
)

top_ips = ip_stats.sort_values(
    ["contas_distintas", "taxa_fraude", "valor_total"],
    ascending=False,
).head(20)

top_ips

,ip_id,qtd_transacoes,contas_distintas,clientes_distintos,beneficiarios_distintos,valor_total,valor_medio,taxa_fraude,uf_origem,tipo_rede,risco_rede
2368,IP_002369,134,92,90,56,"934,404.91","6,973.17",0.70,PE,movel,alto
1996,IP_001997,145,91,91,52,"1,033,160.55","7,125.25",0.77,MG,movel,alto
1100,IP_001101,140,79,79,45,"1,048,210.39","7,487.22",0.81,RS,residencial,alto
1108,IP_001109,144,78,77,39,"1,108,719.68","7,699.44",0.85,BA,vpn_proxy,alto
563,IP_000564,128,75,74,41,"946,469.32","7,394.29",0.80,CE,residencial,alto
2712,IP_002713,124,74,72,43,"920,033.64","7,419.63",0.77,GO,residencial,alto
567,IP_000568,120,71,69,35,"963,521.40","8,029.35",0.84,SP,movel,alto
2422,IP_002423,117,70,70,37,"798,075.46","6,821.16",0.85,SC,movel,alto
1279,IP_001280,115,69,68,33,"1,003,049.49","8,722.17",0.85,RS,residencial,alto
479,IP_000480,101,61,61,34,"839,086.98","8,307.79",0.84,BA,residencial,alto


## 13. Contas com Comportamento Suspeito

A conta é uma entidade central para investigação antifraude.

Nesta etapa, agregamos comportamento por conta para identificar:

- contas com muitas transações;
- contas com alto valor total;
- contas com alta taxa de fraude sintética;
- contas que usam muitos dispositivos;
- contas que enviam para muitos beneficiários;
- contas novas com comportamento transacional relevante.

In [26]:
account_stats = (
    transacoes_enriched
    .groupby("conta_origem_id")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        valor_total=("valor", "sum"),
        valor_medio=("valor", "mean"),
        valor_max=("valor", "max"),
        taxa_fraude=("is_fraud", "mean"),
        dispositivos_distintos=("device_id", "nunique"),
        ips_distintos=("ip_id", "nunique"),
        beneficiarios_distintos=("beneficiario_id", "nunique"),
        primeira_transacao=("data_hora", "min"),
        ultima_transacao=("data_hora", "max"),
        idade_conta_min_dias=("idade_conta_dias", "min"),
    )
    .reset_index()
    .merge(
        contas[
            [
                "conta_id",
                "cliente_id",
                "tipo_conta",
                "data_abertura",
                "status_conta",
                "limite_transacional_diario",
            ]
        ],
        left_on="conta_origem_id",
        right_on="conta_id",
        how="left",
    )
)

top_accounts = account_stats.sort_values(
    ["taxa_fraude", "valor_total", "qtd_transacoes"],
    ascending=False,
).head(30)

top_accounts

,conta_origem_id,qtd_transacoes,valor_total,valor_medio,valor_max,taxa_fraude,dispositivos_distintos,ips_distintos,beneficiarios_distintos,primeira_transacao,ultima_transacao,idade_conta_min_dias,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario
663,CTA_000664,85,"590,316.06","6,944.89","11,984.34",0.93,84,84,84,2025-01-03 07:10:10,2025-12-26 22:12:30,977,CTA_000664,CLI_002370,digital,2022-05-02,ativa,5000
2508,CTA_002509,86,"525,957.06","6,115.78","11,852.33",0.90,85,84,85,2025-01-03 06:16:16,2025-12-31 06:52:31,0,CTA_002509,CLI_004569,corrente,2025-09-21,ativa,5000
4447,CTA_004448,66,"379,933.81","5,756.57","11,883.46",0.88,66,66,66,2025-01-02 06:48:14,2025-12-21 05:35:24,294,CTA_004448,CLI_004188,corrente,2024-03-14,ativa,2000
73,CTA_000074,73,"421,227.98","5,770.25","11,736.61",0.88,72,72,73,2025-01-03 11:05:30,2025-12-16 05:04:11,475,CTA_000074,CLI_000622,pagamento,2023-09-16,ativa,10000
1975,CTA_001976,76,"411,664.39","5,416.64","11,931.22",0.87,76,76,75,2025-01-02 22:59:04,2025-12-29 07:17:53,1352,CTA_001976,CLI_002250,digital,2021-04-21,ativa,2000
161,CTA_000162,75,"403,724.81","5,383.00","11,803.46",0.87,75,73,75,2025-01-03 22:00:41,2025-12-26 10:18:57,515,CTA_000162,CLI_001496,corrente,2023-08-07,ativa,10000
2394,CTA_002395,78,"471,438.75","6,044.09","11,872.99",0.85,77,77,77,2025-01-02 07:42:54,2025-12-29 17:32:43,557,CTA_002395,CLI_004390,corrente,2023-06-25,ativa,2000
1037,CTA_001038,38,"64,304.62","1,692.23","3,414.88",0.84,38,38,38,2025-04-13 16:07:07,2025-12-10 11:02:37,418,CTA_001038,CLI_000779,corrente,2024-02-20,ativa,5000
874,CTA_000875,75,"385,964.26","5,146.19","11,936.05",0.84,74,73,75,2025-01-05 12:27:37,2025-12-30 15:30:59,0,CTA_000875,CLI_003349,digital,2025-08-25,ativa,20000
1133,CTA_001134,79,"433,077.01","5,481.99","11,957.11",0.84,79,79,78,2025-01-05 02:41:05,2025-12-31 15:08:47,0,CTA_001134,CLI_004800,corrente,2025-09-25,ativa,2000


## 14. Candidatos Iniciais a Regras Antifraude

Com base na EDA, serão criadas flags exploratórias que servirão como candidatas para o futuro motor de regras.

Essas flags ainda não representam a versão final das regras. Elas servem para validar se os sinais analíticos estão coerentes com os cenários sintéticos.

Regras candidatas:

- alto valor transacional;
- dispositivo compartilhado;
- beneficiário concentrador;
- conta nova com alto valor;
- IP ou dispositivo de maior risco;
- transações em rajada por conta e hora.

In [27]:
value_p90 = transacoes_enriched["valor"].quantile(0.90)
value_p95 = transacoes_enriched["valor"].quantile(0.95)

device_feature = device_stats[
    ["device_id", "contas_distintas", "taxa_fraude"]
].rename(
    columns={
        "contas_distintas": "device_contas_distintas",
        "taxa_fraude": "device_taxa_fraude",
    }
)

beneficiary_feature = beneficiary_stats[
    ["beneficiario_id", "contas_origem_distintas", "taxa_fraude"]
].rename(
    columns={
        "contas_origem_distintas": "beneficiario_contas_origem_distintas",
        "taxa_fraude": "beneficiario_taxa_fraude",
    }
)

burst_feature = (
    transacoes_enriched
    .assign(data_hora_truncada=transacoes_enriched["data_hora"].dt.floor("h"))
    .groupby(["conta_origem_id", "data_hora_truncada"])
    .agg(qtd_transacoes_conta_hora=("transacao_id", "count"))
    .reset_index()
)

rule_candidates = (
    transacoes_enriched
    .assign(data_hora_truncada=transacoes_enriched["data_hora"].dt.floor("h"))
    .merge(device_feature, on="device_id", how="left")
    .merge(beneficiary_feature, on="beneficiario_id", how="left")
    .merge(burst_feature, on=["conta_origem_id", "data_hora_truncada"], how="left")
)

rule_candidates["r001_alto_valor"] = rule_candidates["valor"] >= value_p95

rule_candidates["r002_dispositivo_compartilhado"] = (
    rule_candidates["device_contas_distintas"] >= 5
)

rule_candidates["r003_beneficiario_concentrador"] = (
    rule_candidates["beneficiario_contas_origem_distintas"] >= 25
)

rule_candidates["r004_conta_nova_alto_valor"] = (
    (rule_candidates["idade_conta_dias"] <= 60)
    & (rule_candidates["valor"] >= value_p90)
)

rule_candidates["r005_rede_ou_device_risco"] = (
    (rule_candidates["risco_rede"] == "alto")
    | (rule_candidates["fingerprint_risco"] == "alto")
)

rule_candidates["r006_transacoes_em_rajada"] = (
    rule_candidates["qtd_transacoes_conta_hora"] >= 5
)

rule_columns = [
    "r001_alto_valor",
    "r002_dispositivo_compartilhado",
    "r003_beneficiario_concentrador",
    "r004_conta_nova_alto_valor",
    "r005_rede_ou_device_risco",
    "r006_transacoes_em_rajada",
]

rule_candidates["qtd_regras_acionadas"] = rule_candidates[rule_columns].sum(axis=1)

rule_candidates[
    [
        "transacao_id",
        "conta_origem_id",
        "valor",
        "data_hora",
        "tipo_transacao",
        "canal",
        "fraud_scenario",
        "is_fraud",
        "qtd_regras_acionadas",
        *rule_columns,
    ]
].head()

,transacao_id,conta_origem_id,valor,data_hora,tipo_transacao,canal,fraud_scenario,is_fraud,qtd_regras_acionadas,r001_alto_valor,r002_dispositivo_compartilhado,r003_beneficiario_concentrador,r004_conta_nova_alto_valor,r005_rede_ou_device_risco,r006_transacoes_em_rajada
0,TX_00007707,CTA_001877,203.30,2025-01-01 03:05:51,pix,app,normal,0,2,False,True,False,False,True,False
1,TX_00072064,CTA_003365,275.99,2025-01-01 03:13:25,pix,app,normal,0,1,False,True,False,False,False,False
2,TX_00075764,CTA_000156,11.59,2025-01-01 04:00:23,pix,app,normal,0,2,False,True,True,False,False,False
3,TX_00023203,CTA_005491,296.97,2025-01-01 04:08:36,pix,app,normal,0,1,False,True,False,False,False,False
4,TX_00001302,CTA_000997,94.98,2025-01-01 04:15:04,boleto,api,normal,0,2,False,True,True,False,False,False


In [28]:
rule_effectiveness_summary = []

for rule in rule_columns:
    temp = (
        rule_candidates
        .groupby(rule)
        .agg(
            qtd_transacoes=("transacao_id", "count"),
            taxa_fraude=("is_fraud", "mean"),
            valor_medio=("valor", "mean"),
            qtd_cenarios=("fraud_scenario", "nunique"),
        )
        .reset_index()
    )
    
    triggered = temp[temp[rule] == True].copy()
    
    if not triggered.empty:
        rule_effectiveness_summary.append(
            {
                "regra_candidata": rule,
                "qtd_transacoes_acionadas": int(triggered["qtd_transacoes"].iloc[0]),
                "taxa_fraude_entre_acionadas": float(triggered["taxa_fraude"].iloc[0]),
                "valor_medio_acionadas": float(triggered["valor_medio"].iloc[0]),
                "qtd_cenarios_cobertos": int(triggered["qtd_cenarios"].iloc[0]),
            }
        )

rule_effectiveness_df = pd.DataFrame(rule_effectiveness_summary).sort_values(
    ["taxa_fraude_entre_acionadas", "qtd_transacoes_acionadas"],
    ascending=False,
)

rule_effectiveness_df

,regra_candidata,qtd_transacoes_acionadas,taxa_fraude_entre_acionadas,valor_medio_acionadas,qtd_cenarios_cobertos
5,r006_transacoes_em_rajada,1183,1.00,"1,871.81",1
3,r004_conta_nova_alto_valor,1944,0.77,"15,944.20",7
0,r001_alto_valor,4000,0.75,"11,708.30",7
4,r005_rede_ou_device_risco,9517,0.31,"1,685.05",7
2,r003_beneficiario_concentrador,31469,0.13,"1,179.11",7
1,r002_dispositivo_compartilhado,79996,0.09,"1,012.58",7


In [29]:
rule_count_summary = (
    rule_candidates
    .groupby("qtd_regras_acionadas")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_fraude=("is_fraud", "mean"),
        valor_medio=("valor", "mean"),
    )
    .reset_index()
    .sort_values("qtd_regras_acionadas")
)

rule_count_summary

,qtd_regras_acionadas,qtd_transacoes,taxa_fraude,valor_medio
0,0,3,0.00,496.48
1,1,40866,0.00,412.73
2,2,32061,0.09,601.19
3,3,5422,0.45,"4,387.09"
4,4,1385,0.93,"12,974.53"
5,5,263,1.00,"11,811.44"


In [30]:
high_priority_cases = (
    rule_candidates
    .loc[rule_candidates["qtd_regras_acionadas"] >= 3]
    .sort_values(
        ["qtd_regras_acionadas", "valor", "device_contas_distintas", "beneficiario_contas_origem_distintas"],
        ascending=False,
    )
    [
        [
            "transacao_id",
            "conta_origem_id",
            "beneficiario_id",
            "device_id",
            "ip_id",
            "valor",
            "data_hora",
            "tipo_transacao",
            "canal",
            "fraud_scenario",
            "is_fraud",
            "qtd_regras_acionadas",
            "device_contas_distintas",
            "beneficiario_contas_origem_distintas",
            "qtd_transacoes_conta_hora",
            "idade_conta_dias",
        ]
    ]
    .head(30)
)

high_priority_cases

,transacao_id,conta_origem_id,beneficiario_id,device_id,ip_id,valor,data_hora,tipo_transacao,canal,fraud_scenario,is_fraud,qtd_regras_acionadas,device_contas_distintas,beneficiario_contas_origem_distintas,qtd_transacoes_conta_hora,idade_conta_dias
20118,TX_00040529,CTA_005726,BEN_003230,DEV_002736,IP_002522,"43,810.56",2025-04-03 20:10:24,ted,internet_banking,new_account_high_value,1,5,76,29,1,0
56934,TX_00023417,CTA_000290,BEN_003007,DEV_000999,IP_002541,"42,646.32",2025-09-21 01:21:27,pix,internet_banking,new_account_high_value,1,5,25,30,1,0
59766,TX_00006881,CTA_003063,BEN_000235,DEV_001468,IP_001235,"41,639.83",2025-10-04 01:43:45,ted,atm,new_account_high_value,1,5,19,28,1,1
32002,TX_00077782,CTA_002140,BEN_000486,DEV_000198,IP_000466,"41,629.43",2025-05-28 11:20:46,pix,api,new_account_high_value,1,5,17,28,1,0
24481,TX_00019227,CTA_004264,BEN_000805,DEV_000210,IP_000869,"40,600.73",2025-04-24 08:10:25,ted,agencia,new_account_high_value,1,5,26,26,1,0
162,TX_00034217,CTA_000199,BEN_002481,DEV_003007,IP_002686,"39,474.81",2025-01-01 20:19:14,ted,app,new_account_high_value,1,5,15,26,1,0
68268,TX_00012530,CTA_005358,BEN_000721,DEV_003719,IP_001080,"38,699.10",2025-11-12 14:06:17,pix,agencia,new_account_high_value,1,5,17,28,1,0
46890,TX_00005821,CTA_003299,BEN_000430,DEV_000552,IP_002978,"38,159.60",2025-08-05 10:28:07,pix,api,new_account_high_value,1,5,21,30,1,0
6982,TX_00079880,CTA_002426,BEN_002525,DEV_001231,IP_001497,"35,840.63",2025-02-02 12:09:32,pix,agencia,new_account_high_value,1,5,15,29,1,0
65961,TX_00053067,CTA_004187,BEN_000667,DEV_002517,IP_002154,"35,328.11",2025-11-01 20:57:41,pix,internet_banking,new_account_high_value,1,5,13,28,1,0


## 15. Síntese dos Achados da EDA

A partir das análises realizadas, serão consolidados os principais achados exploratórios.

Essa síntese será útil para orientar o próximo notebook, dedicado à criação do motor de regras antifraude.

In [31]:
eda_findings = pd.DataFrame(
    [
        {
            "achado": "Cenários sintéticos de fraude foram preservados",
            "evidencia": "A base contém cenários como dispositivo compartilhado, beneficiário concentrador, conta nova de alto valor e rede coordenada.",
            "implicacao": "A base está adequada para desenvolver regras antifraude e análises de grafo.",
        },
        {
            "achado": "Valores transacionais variam fortemente por cenário",
            "evidencia": "Cenários como conta nova de alto valor e beneficiário concentrador apresentam maior valor médio.",
            "implicacao": "Regras baseadas em percentis de valor podem capturar parte dos riscos.",
        },
        {
            "achado": "Beneficiários concentradores possuem alto potencial investigativo",
            "evidencia": "Alguns beneficiários recebem transações de muitas contas distintas.",
            "implicacao": "Beneficiários devem ser nós centrais no Knowledge Graph.",
        },
        {
            "achado": "Dispositivos compartilhados são sinal estrutural forte",
            "evidencia": "Dispositivos com muitas contas distintas aparecem como candidatos relevantes.",
            "implicacao": "Device centrality e shared-device rules devem compor o motor antifraude.",
        },
        {
            "achado": "Combinação de regras aumenta a priorização",
            "evidencia": "Transações com múltiplas regras acionadas apresentam maior valor investigativo.",
            "implicacao": "O próximo notebook deve criar um score baseado em severidade e quantidade de sinais.",
        },
    ]
)

eda_findings

,achado,evidencia,implicacao
0,Cenários sintéticos de fraude foram preservados,"A base contém cenários como dispositivo compartilhado, beneficiário concentrador, conta nova de alto valor e rede coordenada.",A base está adequada para desenvolver regras antifraude e análises de grafo.
1,Valores transacionais variam fortemente por cenário,Cenários como conta nova de alto valor e beneficiário concentrador apresentam maior valor médio.,Regras baseadas em percentis de valor podem capturar parte dos riscos.
2,Beneficiários concentradores possuem alto potencial investigativo,Alguns beneficiários recebem transações de muitas contas distintas.,Beneficiários devem ser nós centrais no Knowledge Graph.
3,Dispositivos compartilhados são sinal estrutural forte,Dispositivos com muitas contas distintas aparecem como candidatos relevantes.,Device centrality e shared-device rules devem compor o motor antifraude.
4,Combinação de regras aumenta a priorização,Transações com múltiplas regras acionadas apresentam maior valor investigativo.,O próximo notebook deve criar um score baseado em severidade e quantidade de sinais.


## 16. Exportação dos Artefatos da EDA

Nesta etapa serão exportados arquivos de apoio para a pasta `docs/`.

Esses artefatos ajudam a documentar o raciocínio analítico do projeto e poderão ser usados posteriormente no README, na página de portfólio e na conclusão executiva.

In [32]:
def safe_markdown_table(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_csv(index=False)


eda_report_lines = [
    "# EDA Transactional Fraud — Resumo Executivo",
    "",
    "Este documento consolida os principais resultados do Notebook 02.",
    "",
    "## KPIs Gerais",
    "",
    safe_markdown_table(kpi_df),
    "",
    "## Distribuição dos Cenários",
    "",
    safe_markdown_table(scenario_distribution),
    "",
    "## Valor por Cenário",
    "",
    safe_markdown_table(value_summary_by_scenario),
    "",
    "## Top Beneficiários Concentradores",
    "",
    safe_markdown_table(top_beneficiaries.head(10)),
    "",
    "## Top Dispositivos Compartilhados",
    "",
    safe_markdown_table(top_devices.head(10)),
    "",
    "## Regras Candidatas",
    "",
    safe_markdown_table(rule_effectiveness_df),
    "",
    "## Achados Principais",
    "",
    safe_markdown_table(eda_findings),
    "",
    "## Observação",
    "",
    "Os resultados são derivados de dados sintéticos criados exclusivamente para fins educacionais, analíticos e de portfólio.",
    "",
]

eda_report_path = DOCS_DIR / "eda_transactional_fraud_summary.md"
eda_report_path.write_text("\n".join(eda_report_lines), encoding="utf-8")

rule_candidates_sample_path = REPORTS_DIR / "rule_candidates_sample.csv"
high_priority_cases.to_csv(rule_candidates_sample_path, index=False, encoding="utf-8")

print(f"Relatório EDA exportado em: {eda_report_path}")
print(f"Amostra de casos prioritários exportada em: {rule_candidates_sample_path}")

Relatório EDA exportado em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\eda_transactional_fraud_summary.md
Amostra de casos prioritários exportada em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\rule_candidates_sample.csv


## 17. Conclusão Executiva do Notebook 02

Este notebook realizou a análise exploratória inicial da base sintética de transações financeiras do projeto **Fraud Graph Analytics**.

A EDA confirmou que a base possui padrões adequados para avançar para a construção de regras antifraude, incluindo:

- cenários de fraude sintética bem definidos;
- diferenças relevantes de valor entre cenários;
- beneficiários com concentração de recebimentos;
- dispositivos utilizados por múltiplas contas;
- IPs e dispositivos com sinal de risco;
- contas com comportamento transacional concentrado;
- transações com múltiplas regras candidatas acionadas.

Os principais sinais identificados serão utilizados no próximo notebook para estruturar um **motor de regras antifraude explicável**, com regras nomeadas, severidade, score parcial e justificativas interpretáveis.

O próximo passo será o:

**Notebook 03 — Rules Engine Antifraud**